# Lekce 11 - Protokol Agent-to-Agent (A2A)


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Co je protokol A2A?

Protokol **Agent-to-Agent (A2A)** je otevřený standard, který umožňuje AI agentům komunikovat,
navzájem se objevovat a spolupracovat – i když jsou postaveni na různých rámcích nebo hostováni
různými službami.

Klíčové koncepty:

- **Objevování** – Agenti zveřejňují *Agent Card*, která popisuje jejich schopnosti, což usnadňuje,
  aby ostatní agenti (nebo orchestrátoři) našli správného specialistu pro úkol.
- **Předávání zpráv** – Agenti si vyměňují strukturované zprávy prostřednictvím společného protokolu,
  takže žádost od jednoho agenta může pochopit a splnit jiný bez ohledu na jeho vnitřní
  implementaci.
- **Životní cyklus úkolu** – A2A definuje stavy jako *předáno*, *pracuje se na tom*, *dokončeno* a
  *selhalo*, což orchestrátorovi umožňuje plný přehled o tom, jak postupuje delegovaný úkol.

V této lekci simulujeme spolupráci ve stylu A2A tím, že propojujeme tři specializované cestovní agenty
do pracovního postupu, kde každý agent přispívá svými odbornými znalostmi a předává výsledky dalšímu.


## Vytváření specializovaných cestovních agentů


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Spolupráce více agentů pomocí pracovního postupu

Propojíme tři agenty do sekvenčního pracovního postupu, který odráží přenos zpráv A2A:

1. **CurrencyExchangeAgent** přijímá uživatelskou žádost a poskytuje pokyny k měnám.
2. **ActivityPlannerAgent** přijímá zlepšený kontext a přidává doporučení aktivit.
3. **TravelManagerAgent** syntetizuje oba vstupy do konečného cestovního shrnutí.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Pochopení A2A ve výrobním prostředí

V produkčním prostředí protokol A2A odemyká silné scénáře napříč službami:

| Možnost | Popis |
|---|---|
| **Interop napříč rámci** | Agent vytvořený v jednom rámci může delegovat úkoly agentovi vytvořenému v jiném A2A-kompatibilním rámci, což umožňuje skutečnou interoperabilitu napříč organizacemi. |
| **Hranice služeb** | Agenti mohou žít v samostatných mikroslužbách, cloudových regionech nebo dokonce v různých organizacích a přitom bezproblémově spolupracovat. |
| **Dynamické vyhledávání** | Orchestrátor může za běhu dotazovat registr Agent Card a najít tak nejvhodnějšího specialistu pro daný podúkol. |
| **Streaming a push notifikace** | A2A podporuje Server-Sent Events (SSE) pro aktualizace průběhu v reálném čase a push notifikace pro dlouhotrvající úkoly. |

Pracovní postup, který jsme výše vytvořili, je zjednodušená, v-procesní verze tohoto vzoru. V reálném
nasazení by každý agent vystavil HTTP endpoint, publikoval Agent Card a komunikoval
pomocí protokolu A2A JSON-RPC.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
